# High-Frequency Trading Example

This notebook demonstrates how to use high-frequency trading capabilities to implement low-latency trading strategies.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import asyncio

from src.strategies.high_frequency_trader import (
    HighFrequencyTrader,
    SignalType,
    MarketSignal,
    ExecutionMetrics
)
from src.data.binance_fetcher import BinanceFetcher

## Configuration

Load and configure high-frequency trading settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize trader
trader = HighFrequencyTrader(
    config=config,
    exchange='binance',
    max_position=1.0,
    risk_limit=0.02,
    signal_threshold=0.5,
    execution_timeout=0.1,
    buffer_size=1000
)

print("Trader initialized with:")
print(f"Max position: {trader.max_position}")
print(f"Risk limit: {trader.risk_limit}")
print(f"Signal threshold: {trader.signal_threshold}")
print(f"Execution timeout: {trader.execution_timeout}s")

## Market Data Analysis

Analyze real-time market data and signal generation.

In [ ]:
async def analyze_market_data():
    """Analyze market data and signals."""
    symbol = 'BTC/USDT'
    data_points = []
    signals = []
    
    # Collect data for 10 seconds
    start_time = time.time()
    while time.time() - start_time < 10:
        # Update market data
        price, volume, order_flow = await trader.update_market_data(symbol)
        
        # Generate signals
        momentum = trader.generate_momentum_signal()
        mean_rev = trader.generate_mean_reversion_signal()
        flow = trader.generate_order_flow_signal()
        
        # Record data
        data_points.append({
            'timestamp': datetime.now(),
            'price': price,
            'volume': volume,
            'imbalance': order_flow['imbalance'],
            'spread': order_flow['spread']
        })
        
        # Record signals
        if momentum:
            signals.append({
                'timestamp': datetime.now(),
                'type': 'momentum',
                'direction': momentum.direction,
                'strength': momentum.strength
            })
        if mean_rev:
            signals.append({
                'timestamp': datetime.now(),
                'type': 'mean_reversion',
                'direction': mean_rev.direction,
                'strength': mean_rev.strength
            })
        if flow:
            signals.append({
                'timestamp': datetime.now(),
                'type': 'order_flow',
                'direction': flow.direction,
                'strength': flow.strength
            })
        
        await asyncio.sleep(0.001)  # 1ms interval
    
    return pd.DataFrame(data_points), pd.DataFrame(signals)

# Analyze market data
print("Analyzing market data for 10 seconds...")
data_df, signals_df = await analyze_market_data()

print(f"\nCollected {len(data_df)} data points")
print(f"Generated {len(signals_df)} signals")

# Plot analysis
plt.figure(figsize=(15, 10))

# Price and volume
plt.subplot(2, 2, 1)
ax1 = plt.gca()
ax2 = ax1.twinx()
ax1.plot(data_df['timestamp'], data_df['price'], 'b-', label='Price')
ax2.plot(data_df['timestamp'], data_df['volume'], 'r-', alpha=0.5, label='Volume')
ax1.set_ylabel('Price', color='b')
ax2.set_ylabel('Volume', color='r')
plt.title('Price and Volume')

# Order book metrics
plt.subplot(2, 2, 2)
plt.plot(data_df['timestamp'], data_df['imbalance'], label='Imbalance')
plt.plot(data_df['timestamp'], data_df['spread'], label='Spread')
plt.title('Order Book Metrics')
plt.legend()

# Signal distribution
plt.subplot(2, 2, 3)
signals_df['type'].value_counts().plot(kind='bar')
plt.title('Signal Distribution')

# Signal strength
plt.subplot(2, 2, 4)
sns.boxplot(x='type', y='strength', data=signals_df)
plt.title('Signal Strength Distribution')

plt.tight_layout()
plt.show()

## Signal Analysis

Analyze signal generation and combination.

In [ ]:
def analyze_signals(signals_df: pd.DataFrame):
    """Analyze trading signals."""
    # Signal timing analysis
    plt.figure(figsize=(15, 5))
    
    # Signal timing
    plt.subplot(1, 2, 1)
    for signal_type in signals_df['type'].unique():
        signals = signals_df[signals_df['type'] == signal_type]
        plt.scatter(signals['timestamp'],
                   signals['direction'] * signals['strength'],
                   label=signal_type, alpha=0.5)
    plt.title('Signal Timing')
    plt.legend()
    
    # Signal correlation
    plt.subplot(1, 2, 2)
    pivot = signals_df.pivot(columns='type',
                            values='direction').fillna(0)
    sns.heatmap(pivot.corr(), annot=True, cmap='RdYlBu')
    plt.title('Signal Correlation')
    
    plt.tight_layout()
    plt.show()
    
    # Print signal statistics
    print("\nSignal Statistics:")
    for signal_type in signals_df['type'].unique():
        signals = signals_df[signals_df['type'] == signal_type]
        print(f"\n{signal_type}:")
        print(f"Count: {len(signals)}")
        print(f"Avg Strength: {signals['strength'].mean():.4f}")
        print(f"Direction Bias: {signals['direction'].mean():.4f}")

# Analyze signals
analyze_signals(signals_df)

## Execution Analysis

Analyze order execution and performance metrics.

In [ ]:
async def analyze_execution(duration: int = 30):
    """Analyze execution performance."""
    print(f"Running strategy for {duration} seconds...")
    
    # Run strategy
    try:
        async def stop_after_delay():
            await asyncio.sleep(duration)
            raise KeyboardInterrupt
        
        await asyncio.gather(
            trader.run('BTC/USDT', interval=0.001),
            stop_after_delay()
        )
    except KeyboardInterrupt:
        pass
    
    # Analyze metrics
    metrics = trader.get_performance_metrics()
    
    print("\nPerformance Metrics:")
    print(f"Average Latency: {metrics['avg_latency']*1000:.2f}ms")
    print(f"Maximum Latency: {metrics['max_latency']*1000:.2f}ms")
    print(f"Fill Rate: {metrics['avg_fill_rate']*100:.1f}%")
    print(f"Number of Trades: {metrics['num_trades']}")
    
    # Plot metrics
    execution_df = pd.DataFrame([
        {
            'timestamp': datetime.fromtimestamp(m.timestamp),
            'latency': m.latency,
            'slippage': m.slippage,
            'fill_rate': m.fill_rate,
            'cost': m.cost
        }
        for m in trader.metrics
    ])
    
    if not execution_df.empty:
        plt.figure(figsize=(15, 10))
        
        # Latency
        plt.subplot(2, 2, 1)
        plt.plot(execution_df['timestamp'],
                execution_df['latency'] * 1000)
        plt.title('Execution Latency (ms)')
        
        # Slippage
        plt.subplot(2, 2, 2)
        plt.plot(execution_df['timestamp'],
                execution_df['slippage'] * 10000)
        plt.title('Slippage (bps)')
        
        # Fill rate
        plt.subplot(2, 2, 3)
        plt.plot(execution_df['timestamp'],
                execution_df['fill_rate'] * 100)
        plt.title('Fill Rate (%)')
        
        # Costs
        plt.subplot(2, 2, 4)
        plt.plot(execution_df['timestamp'],
                execution_df['cost'].cumsum())
        plt.title('Cumulative Costs')
        
        plt.tight_layout()
        plt.show()

# Analyze execution
await analyze_execution()

## Performance Analysis

Analyze overall strategy performance.

In [ ]:
def analyze_performance():
    """Analyze strategy performance."""
    # Calculate returns
    position_changes = np.diff(
        [0] + [o['size'] for o in trader.filled_orders]
    )
    prices = [o['price'] for o in trader.filled_orders]
    returns = position_changes * prices
    
    # Calculate metrics
    total_pnl = sum(returns)
    sharpe = np.mean(returns) / np.std(returns) if len(returns) > 1 else 0
    win_rate = np.mean(returns > 0) if returns else 0
    
    print("Performance Metrics:")
    print(f"Total P&L: ${total_pnl:,.2f}")
    print(f"Sharpe Ratio: {sharpe:.2f}")
    print(f"Win Rate: {win_rate*100:.1f}%")
    
    # Plot performance
    plt.figure(figsize=(15, 5))
    
    # Cumulative P&L
    plt.subplot(1, 2, 1)
    plt.plot(np.cumsum(returns))
    plt.title('Cumulative P&L')
    
    # Return distribution
    plt.subplot(1, 2, 2)
    sns.histplot(returns)
    plt.title('Return Distribution')
    
    plt.tight_layout()
    plt.show()

# Analyze performance
analyze_performance()